## Problem Statement

Small and medium-sized enterprises (SMEs) in Kenya, particularly in agrovet, retail,
pharmacy, and electronics segments, routinely lose revenue to two interlinked inventory
problems: unexplained stock shrinkage and unpredictable stockouts. These businesses
typically lack the tooling that larger retailers use, such as demand forecasting or automated
reorder triggers, and instead rely on manual stock counts and fixed reorder levels that don't
adapt to product-specific demand patterns or supplier lead times.

To explore how data analysis and machine learning could address this gap, | built a synthetic
dataset engineered to reflect realistic SME operating conditions, including common realworld data quality issues (missing values, inconsistent formatting, mixed date types),
seasonal demand patterns (e.g. Kenyan planting seasons driving fertiliser demand), and
category-specific shrinkage behavior. The dataset was designed based on domain research
rather than drawn from live business records, so the specific figures are illustrative, but the
data cleaning challenges, feature engineering choices, and modeling approach reflect what a
real analysis would require.

The analysis set out to answer three questions: where in a business shrinkage and stockout
risk concentrate, which measurable signals most reliably predict stockout risk, and whether a
predictive model could flag at-risk products early enough to inform reorder decisions.

The project covers the full analytical pipeline, from data cleaning and standardization,
through exploratory analysis using a structured 5SW1H framework (who, what, when, where,
why, how), to feature engineering (rolling demand averages, shrinkage rates, days-of-stockremaining) and model development. Four classification models (Logistic Regression, Random
Forest, XGBoost, LightGBM) were trained and compared, with XGBoost selected as the best
performer based on ROC-AUC (0.9993 on the held-out test set). 


In [ ]:

import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import joblib
import json
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import os
import optuna
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    f1_score, precision_score, recall_score, RocCurveDisplay,
    precision_recall_curve, average_precision_score
)
from sklearn.calibration import CalibratedClassifierCV
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import plotly.io as pio

pio.renderers.default = "png"

#  ------Plot style -------------------------------------------------------------------
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')
FIGSIZE = (14, 5)
COLORS = {'Agrovet':'#1D9E75','Mini Supermarket':'#7F77DD',
'Pharmacy':'#D85A30','Electronics':'#378ADD','Wholesaler':'#BA7517'}

print(f"Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"pandas {pd.__version__} | numpy {np.__version__} | seaborn {sns.__version__}")

OUTPUT_DIR = "stockout_model_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:

inventory_df = pd.read_csv(r"D:\SME restock and stock prediction\kenya_sme_inventory_dirty_2024_2026.csv")
df = inventory_df.copy()

In [ ]:
# inventory_df.info()
df.duplicated().sum()

#drop duplicates
df.drop_duplicates(inplace=True)

In [ ]:
#Standardize
df.info()

df['date'] = pd.to_datetime(df['date'], format='mixed',errors='coerce')

In [ ]:

df.head()

In [ ]:

df.shape

In [ ]:

missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if missing.empty:
    print("No missing values - data is clean.")
else:
    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.barh(missing.index, missing.values, color='#D85A30', height=0.6)
    ax.bar_label(bars, fmt='{:,.0f}', padding=4, fontsize=10)
    ax.set_xlabel('Missing count')
    ax.set_title('Missing values per column', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print(missing)

In [ ]:

#standardazation
df['product_name'] = df['product_name'].str.strip()
df['category']=df['category'].str.strip()

CATEGORY={
'flour & grains':['Flour & Grains', 'flour & grains', 'FLOUR & GRAINS'],
'dairy':['Dairy','dairy', 'DAIRY'],
'oils & fats':['Oils & Fats', 'OILS & FATS', 'oils & fats'],
'sugar & spices':['Sugar & Spices', 'SUGAR & SPICES', 'sugar & spices'],
'detergents':['Detergents', 'detergents', 'DETERGENTS'],
'seasonings':['Seasonings', 'SEASONINGS','seasonings'],
'analgesics':['Analgesics', 'analgesics', 'ANALGESICS'],
'antibiotics':['Antibiotics', 'antibiotics', 'ANTIBIOTICS'],
'rehydration':['REHYDRATION','rehydration', 'Rehydration'],
'antiseptics':['Antiseptics', 'antiseptics','ANTISEPTICS'],
'diagnostics':['Diagnostics', 'diagnostics', 'DIAGNOSTICS'],
'fertilisers':['Fertilisers', 'FERTILISERS', 'fertilisers'],
'seeds':['Seeds', 'SEEDS','seeds'],
'veterinary':['Veterinary', 'VETERINARY', 'veterinary'],
'accessories':['Accessories','accessories', 'ACCESSORIES']
}

repair_category ={
    value:key
    for key,values in CATEGORY.items()
    for value in values
}

df['category'] = df['category'].replace(repair_category)

bs_type = {'mini supermarket':['Mini Supermarket', 'Mini Supermarkt', 'MINI SUPERMARKET','Mini Sup.',
'mini supermarket'],
'pharmacy':['Pharmacy', 'Pharamcy','PHARMACY', 'pharmacy', 'Pharmcy'],
'agrovet':['Agrovet', 'Agrovet ','AGROVET', 'agrovet', 'Agrovett'],
'electronics':['ELECTRONICS', 'Electronics','Electroncis', 'electronics']}

repair_bs = {
    value:key
    for key,values in bs_type.items()
    for value in values
}

df['business_type'] = df['business_type'].replace(repair_bs)

unit ={'bags':['bag', 'Bag', 'BAG', 'Bags','Bag '],
'pcs':['Pcs', 'piece', 'pcs', 'PCS'],
'btl':['btl','BOTTLE', 'bottle', 'Btl','Bottle'],
'pck':['pack', 'pck', 'Packs', 'PACK','Pack'],
'strips':['Strips'],
'vial':['VIAL']}

repair_unit = {
    value:key
    for key,values in unit.items()
    for value in values
}

df['unit_of_measure'] = df['unit_of_measure'].replace(repair_unit)

county ={'Nairobi':['Nairobi ', 'NAIROBI ', 'nairobi ', 'NAIROBI', 'nairobi','Nairobi','Nairob','nairob','NAIROB'],
'Kiambu':['Kiambu ', 'kiambu ', 'KIAMBU ', 'kiambu', 'KIAMBU','Kiambu'],
'Kisumu':['Kisumu', 'kisumu', 'KISUMU'],
'Nakuru':['Nakurru', 'nakurru', 'NAKURRU']}

repair_county = {
    value:key
    for key,values in county.items()
    for value in values
}

df['supplier_county'] = df['supplier_county'].replace(repair_county)

In [ ]:

df['supplier_name'].unique()

In [ ]:

# handling missing values
df = df.sort_values(['product_id','date'])
df['opening_stock']= df.groupby('product_id')['opening_stock'].transform(lambda x: x.ffill().bfill())
df['units_sold'] = df.groupby('product_id')['units_sold'].transform(lambda y:y.ffill().bfill())
df['units_sold'] =df['units_sold'].fillna(0)

df['selling_price_kes'] =df.groupby('product_id')['selling_price_kes'].transform(lambda z:z.ffill().bfill())
df['selling_price_kes'] = df['selling_price_kes'].fillna(0)

# Build a lookup from category -> most common supplier in this dataset
category_supplier = (
    df.groupby("category")["supplier_name"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else "Unknown")
    .to_dict()
)

category_county = (
    df.groupby("category")["supplier_county"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else "Unknown")
    .to_dict()
)

# Fill missing supplier_name from category lookup
df["supplier_name"] = df.apply(
    lambda r: category_supplier.get(r["category"], "Unknown")
    if pd.isna(r["supplier_name"]) else r["supplier_name"],
    axis=1
)

# Fill missing supplier_county the same way
df["supplier_county"] = df.apply(
    lambda r: category_county.get(r["category"], "Unknown")
    if pd.isna(r["supplier_county"]) else r["supplier_county"],
    axis=1
)

print(f"supplier_name nulls remaining : {df['supplier_name'].isna().sum()}")
print(f"supplier_county nulls remaining: {df['supplier_county'].isna().sum()}")

In [ ]:

df.isnull().sum()

In [ ]:

#univeriate analysis
df.info()

In [ ]:

df.describe().T

In [ ]:

df.select_dtypes(exclude = 'object').columns

In [ ]:

df['opening_stock']=df['opening_stock'].astype('int')

In [ ]:

col = ['opening_stock', 'units_sold', 'shrinkage_units',
       'units_restocked', 'closing_stock', 'unit_cost_kes',
       'selling_price_kes', 'supplier_lead_days', 'reorder_level',
       'is_holiday', 'is_weekend']

for i in col:
    plt.figure(figsize=(6,4))
    sns.histplot(data = df,x = i,kde=True)
    plt.title(f"{i} distribution")
    plt.xlabel(f"{i}")
    plt.show()

In [ ]:

# col = ['opening_stock', 'units_sold', 'shrinkage_units',
#        'units_restocked', 'closing_stock', 'unit_cost_kes',
#        'selling_price_kes', 'supplier_lead_days', 'reorder_level',
#        'is_holiday', 'is_weekend']

# for i in col:
#     Q1 = df[i].quantile(0.25)
#     Q3 = df[i].quantile(0.75)
#     IQR = Q3 - Q1
#     lower_bound = Q1 - 1.5 * IQR
#     upper_bound = Q3 + 1.5 * IQR

#     outliers = df[
#         (df[i] < lower_bound) |
#         (df[i] > upper_bound)
#     ]
#     outliers
#     df[i] = df[i].clip(lower_bound, upper_bound)

In [ ]:

# df[i] = df[i].clip(lower_bound, upper_bound)

In [ ]:

plt.figure(figsize=(6,4))
sns.boxplot(data = df,x = 'selling_price_kes')
plt.title(f"selling_per_price distribution")
plt.show()

In [ ]:

#univerate analysis
df.info()

In [ ]:

# Gross Margin (revenue - cost per unit sold)
df['gross_margin'] = (df['selling_price_kes'] - df['unit_cost_kes']) * df['units_sold']

# Margin % = gross margin / revenue
df['margin_pct'] = (
    (df['selling_price_kes'] - df['unit_cost_kes']) / df['selling_price_kes'] * 100
).round(2)

# Days of stock remaining = closing_stock / avg daily sales (avoid div/0)
avg_daily_sales = df.groupby('product_id')['units_sold'].transform('mean')
df['days_of_stock_remaining'] = (df['closing_stock' ] / avg_daily_sales.replace(0, float('nan'))).round(1)

# 7-day moving average sales (per product, sorted by date)
df = df.sort_values(['product_id', 'date'])
df['ma_sales_7d'] = (
    df.groupby('product_id')['units_sold']
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
    .round(2)
)

avg_demand = df["ma_sales_7d"].replace(0, 0.1)

df["days_of_stock_remaining"] = (
    df["closing_stock"] / avg_demand
).round(1).clip(0, 365)

df["stockout_risk"] = (
    df["days_of_stock_remaining"] < df["supplier_lead_days"]
).astype(int)

df["needs_restock"] = (
    df["closing_stock"] <= df["reorder_level"]
).astype(int)

df["risk_score"] = df["stockout_risk"] + df["needs_restock"]

df["risk_label"] = df["risk_score"].map({
    0: "Safe",
    1: "Caution",
    2: "Critical"
})

df["recommended_reorder_qty"] = np.where(
    df["stockout_risk"] == 1,
    ((df["ma_sales_7d"] * df["supplier_lead_days"]) + df["reorder_level"]).round(0).astype(int),
    0
)

# 30-day moving average sales
df['ma_sales_30d'] = (
    df.groupby('product_id')['units_sold']
    .transform(lambda x: x.rolling(30, min_periods=1).mean())
    .round(2)
)

# Shrinkage value in KES
df['shrinkage_value_kes'] = df['shrinkage_units'] * df['unit_cost_kes']

# Stock Turnover Rate = units_sold / avg inventory
# avg inventory = (opening_stock + closing_stock) / 2
df['avg_inventory'] = (df['opening_stock'] + df['closing_stock']) / 2
df['stock_turnover_rate'] = (
    df['units_sold'] / df['avg_inventory'].replace(0, float('nan'))
).round(4)

In [ ]:

numeric_cols = ['opening_stock','units_sold','shrinkage_units','closing_stock',
                'unit_cost_kes','selling_price_kes','gross_margin_kes','margin_pct',
                'days_of_stock_remaining','ma_sales_7d','ma_sales_30d',
                'shrinkage_value_kes','stock_turnover_rate']

existing = [c for c in numeric_cols if c in df.columns]
df[existing].describe().round(2).T.style.background_gradient(cmap='Blues', subset=['mean','50%'])

In [ ]:

#distribution of categorical data
pd.set_option('display.max_row',None)
col = df.select_dtypes(include='object')
for i in col:
    print(f"--- Value Counts for: {i} ---")
    print(df[i].value_counts())
    print("\n")

# Section 1 WHO: Which business type and supplier has the most pain?

***Business question***: Where should we focus intervention first?

### 1.1 Shrinkage value by business type 

In [ ]:

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Total shrinkage loss (KES) by business type',
                     'Shrinkage events count by business type'])

biz_shrink = (df.groupby('business_type')['shrinkage_value_kes']
              .sum().sort_values(ascending=False).reset_index())
biz_count = (df.groupby('business_type')['shrinkage_units']
             .sum().sort_values(ascending=False).reset_index())

fig.add_trace(go.Bar(
    x=biz_shrink['business_type'], y=biz_shrink['shrinkage_value_kes'],
    marker_color=[COLORS.get(b,'#D85A30') for b in biz_shrink['business_type']],
    text=biz_shrink['shrinkage_value_kes'].apply(lambda v: f'KES {v:,.0f}'),
    textposition='outside'), row=1, col=1)

fig.add_trace(go.Bar(
    x=biz_count['business_type'], y=biz_count['shrinkage_units'],
    marker_color=[COLORS.get(b,'#D85A30') for b in biz_count['business_type']],
    text=biz_count['shrinkage_units'].apply(lambda v: f'{v:,.0f} units'),
    textposition='outside'), row=1, col=2)

fig.update_layout(height=420, showlegend=False,
    title_text='<b>Shrinkage burden by business type<b>', title_font_size=14)
fig.show()

print("\nTop business type by shrinkage loss:")
print(biz_shrink.to_string(index=True))

In [ ]:
%pip install --upgrade Kaleido 

### 1.2 Stockout risk by business type 

In [ ]:

if 'stockout_risk' in df.columns:
    stockout = (df.groupby('business_type')['stockout_risk']
                .agg(['sum','mean']).reset_index()
                .rename(columns={'sum':'total_events','mean':'risk_rate'}))
    stockout['risk_rate'] = (stockout['risk_rate']*100).round(1)

    fig = px.bar(stockout, x='business_type', y='total_events',
        color='business_type', color_discrete_map=COLORS,
        text='total_events',
        labels={'total_events':'Stockout risk events','business_type':'Business type'},
        title='Stockout risk events by business type')
    fig.update_traces(textposition='outside')
    fig.update_layout(showlegend=False, height=500)
    fig.show()

    print("\nStockout risk rate (% of days at risk):")
    print(stockout[['business_type','total_events','risk_rate']].to_string(index=False))

### 1.3 Top 10 products by total shrinkage loss 


In [ ]:

top_products = (df.groupby(['product_name','business_type'])['shrinkage_value_kes']
                .sum().reset_index()
                .sort_values('shrinkage_value_kes', ascending=False)
                .head(10))

fig = px.bar(top_products, x='shrinkage_value_kes', y='product_name',
    color='business_type', color_discrete_map=COLORS,
    orientation='h',
    text=top_products['shrinkage_value_kes'].apply(lambda v: f'Kes {v:,.0f}'),
    labels={'shrinkage_value_kes':'Total shrinkage loss (KES)','product_name':''},
    title='Top 10 products by shrinkage loss value')
fig.update_traces(textposition='outside')
fig.update_layout(height=450, showlegend=True)
fig.show()

### 1.4 Supplier responsibility - which suppli inked to most restock events? 


In [ ]:

if 'supplier_name' in df.columns:
    sup = (df[df.get('needs_restock', pd.Series(0, index=df.index))==1]
           .groupby('supplier_name')
           .size().reset_index(name='restock_events')
           .sort_values('restock_events', ascending=False)
           .head(10))

    fig = px.bar(sup, x='restock_events', y='supplier_name', orientation='h',
        text='restock_events',
        labels={'restock_events':'Restock trigger events','supplier_name':''},
        title=' Suppliers linked to most restock events')
    fig.update_traces(marker_color='#7F77DD', textposition='outside')
    fig.update_layout(height=380)
    fig.show()

---

# Section 2 WHAT: What is the scale of the inventory problem?
***Business question***: How much money is being lost and where is it going? 


In [ ]:

df.info()

### 2.1 KPl summary- the headline numbers 


In [ ]:

df['revenue'] = (df['units_sold']* df['unit_cost_kes'])

total_shrinkage_kes = df['shrinkage_value_kes'].sum()
total_revenue_kes = df['revenue'].sum() if 'revenue' in df.columns else 0
avg_margin = df['margin_pct'].mean() if 'margin_pct' in df.columns else 0
stockout_events = df['stockout_risk'].sum() if 'stockout_risk' in df.columns else 0
dead_stock_products = (df.groupby('product_id')['units_sold']
                        .sum().eq(0).sum())

print("=" * 55)
print(" SME INVENTORY -- KEY PERFORMANCE INDICATORS")
print("=" * 55)
print(f" Total shrinkage loss    : KES {total_shrinkage_kes:>12,.0f}")
print(f" Total revenue           : KES {total_revenue_kes:>12,.0f}")
print(f" Shrinkage as % revenue  : {total_shrinkage_kes/max(total_revenue_kes,1)*100:>10.2f}%")
print(f" Average gross margin    : {avg_margin:>10.1f}%")
print(f" Stockout risk events    : {stockout_events:>10,.0f}")
print(f" Dead stock products     : {dead_stock_products:>10}")
print("=" * 55)

### 2.2 Shrinkage by category: where is money leaking? 

In [ ]:

cat_shrink = (df.groupby('category')
    .agg(total_loss_kes=('shrinkage_value_kes','sum'),
         avg_daily_loss=('shrinkage_value_kes','mean'),
         shrinkage_days=('shrinkage_units', lambda x: (x>0).sum()))
    .sort_values('total_loss_kes', ascending=False)
    .reset_index())

fig = px.treemap(cat_shrink, path=['category'],
    values='total_loss_kes',
    color='avg_daily_loss',
    color_continuous_scale='RdYlGn_r',
    title='Shrinkage loss distribution by category (treemap)',
    labels={'total_loss_kes':'Total loss (KES)','avg_daily_loss':'Avg daily loss (KES)'})
fig.update_layout(height=450)
fig.show()

print("\nShrinkage by category:")
print(cat_shrink.to_string(index=False))

### 2.3 Dead stock detection products with no sales 


In [ ]:

# Rolling 14-day window -- flag days where cumulative sales = 0
df_sorted = df.sort_values(['product_id','date'])
df_sorted['rolling_14d_sales'] = (df_sorted.groupby('product_id')['units_sold']
    .transform(lambda x: x.rolling(60, min_periods=1).sum()))

dead_stock_days = (df_sorted[df_sorted['rolling_14d_sales']==0]
    .groupby(['product_name','business_type'])
    .size().reset_index(name='dead_stock_days')
    .sort_values('dead_stock_days', ascending=False)
    .head(15))

if not dead_stock_days.empty:
    fig = px.bar(dead_stock_days, x='dead_stock_days', y='product_name',
        color='business_type', color_discrete_map=COLORS, orientation='h',
        text='dead_stock_days',
        title='Dead stock: days with zero sales in 60-day window',
        labels={'dead_stock_days':'Days of zero sales','product_name':''})
    fig.update_traces(textposition='outside')
    fig.update_layout(height=440)
    fig.show()
else:
    print("No dead stock detected -- all products have sales activity")

### 2.4 Gross margin distribution by category 


In [ ]:

if 'margin_pct' in df.columns:
    bar_df = (
        df[df['margin_pct'].between(0, 80)]
        .groupby(['category', 'business_type'], as_index=False)['margin_pct']
        .mean()
    )

    fig = px.bar(
        bar_df,
        x='category', y='margin_pct',
        color='business_type', color_discrete_map=COLORS,
        barmode='group',
        title='Mean gross margin % by category',
        labels={'margin_pct': 'Mean gross margin %', 'category': 'Category'},
    )
    fig.update_layout(height=440, xaxis_tickangle=-30)
    fig.add_hline(y=10, line_dash='dash', line_color='red',
                  annotation_text='10% minimum viable margin',
                  annotation_position='top left')
    fig.show()

---

# Section 3 WHEN: When do stockouts and demand spikes occur?

***Business question***: Are there predictable timing patterns we can build into the ML model?

### 3.1 Monthly sales trend:all products 

In [ ]:

monthly = (df.groupby(df['date'].dt.to_period('M'))
    .agg(total_sales=('units_sold','sum'),
         total_revenue=('revenue_kes','sum') if 'revenue_kes' in df.columns else ('units_sold','sum'),
         stockout_events=('stockout_risk','sum') if 'stockout_risk' in df.columns else ('units_sold','count'))
    .reset_index())
monthly['date'] = monthly['date'].dt.to_timestamp()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
    subplot_titles=['Monthly total units sold','Monthly stockout risk events'],
    vertical_spacing=0.12)

fig.add_trace(go.Scatter(x=monthly['date'], y=monthly['total_sales'],
    mode='lines+markers', name='Units sold',
    line=dict(color='#7F77DD', width=2)), row=1, col=1)

fig.add_trace(go.Bar(x=monthly['date'], y=monthly['stockout_events'],
    name='Stockout events', marker_color='#D85A30'), row=2, col=1)

fig.update_layout(height=520, title='Monthly demand and stockout trend (2024-2026)',
    title_font_size=14, showlegend=True)
fig.show()

### 3.2 Salary-day effect: does end-of-month spike demand? 

In [ ]:

df['is_month_end'] =df['date'].dt.is_month_end.astype(int)

if 'is_month_end' in df.columns:
    me = df.groupby('is_month_end').agg(
        avg_units_sold=('units_sold','mean'),
        avg_stockout_risk=('stockout_risk','mean') if 'stockout_risk' in df.columns else ('units_sold','count')
    ).reset_index()
    me['is_month_end'] = me['is_month_end'].map({0:'Normal days (1-24)','1':'Month end (25-31)',
                                                    1:'Month end (25-31)'})
    me['avg_units_sold'] = me['avg_units_sold'].round(2)

    fig = px.bar(me, x='is_month_end', y='avg_units_sold',
        color='is_month_end',
        color_discrete_sequence=['#7F77DD','#D85A30'],
        text='avg_units_sold',
        title='Salary-day effect: avg daily units sold',
        labels={'avg_units_sold':'Avg units sold per day','is_month_end':''})
    fig.update_traces(textposition='outside')
    fig.update_layout(height=500, showlegend=False)
    fig.show()

    lift = me.set_index('is_month_end')['avg_units_sold']
    keys = list(lift.index)
    if len(keys) >= 2:
        pct = (lift[keys[1]] / lift[keys[0]] - 1) * 100
        print(f"Month-end demand lift: {pct:.1f}% vs normal days")

### 3.3 Kenyan planting season: fertiliser demand pattern 


In [ ]:

fert = df[df['category']=='fertilisers'].copy()

if not fert.empty:
    fert_monthly = (fert.groupby(fert['date'].dt.month)
        ['units_sold'].mean().reset_index()
        .rename(columns={'date':'month','units_sold':'avg_units_sold'}))
    fert_monthly['month_name'] = pd.to_datetime(fert_monthly['month'], format='%m').dt.strftime('%b')
    fert_monthly['is_planting'] = fert_monthly['month'].isin([3,4,9,10])

    fig = px.bar(fert_monthly, x='month_name', y='avg_units_sold',
        color='is_planting',
        color_discrete_map={True:'#1D9E75', False:'#B4B2A9'},
        text=fert_monthly['avg_units_sold'].round(1),
        title='Fertiliser demand by month (green = planting season)',
        labels={'avg_units_sold':'Avg daily units sold','month_name':'Month'})
    fig.update_traces(textposition='outside')
    fig.update_layout(height=400, showlegend=False)
    fig.show()

    print("Planting seasons (Mar-Apr, Sep-Oct) highlighted in green")
else:
    print("No Fertilisers category in dataset")

### 3.4 Day-of-week demand heatmap 


In [ ]:

df['day_name'] = df['date'].dt.strftime('%A')
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

pivot = (df.groupby(['day_name','category'])['units_sold']
    .mean().unstack(fill_value=0)
    .reindex(day_order))

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot, ax=ax, cmap='YlOrRd', fmt='.1f', annot=True,
    linewidths=0.5, cbar_kws={'label':'Avg units sold'})
ax.set_title('Average daily sales by day of week and category', fontsize=13, fontweight='bold')
ax.set_ylabel('')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

### 3.5 Holiday vs normal day sales comparison 

In [ ]:

# Derive is_holiday if not already in df
if 'is_holiday' not in df.columns:
    df['date'] = pd.to_datetime(df['date'])

    # Kenya public holidays (fixed-date + observed)
    kenya_holidays = [
        # 2024
        '2024-01-01',  # New Year's Day
        '2024-04-07',  # Easter Sunday (approximate - adjust per year)
        '2024-04-19',  # Good Friday
        '2024-05-01',  # Labour Day
        '2024-06-01',  # Madaraka Day
        '2024-10-10',  # Huduma Day
        '2024-10-20',  # Mashujaa Day
        '2024-12-12',  # Jamhuri Day
        '2024-12-25',  # Christmas Day
        '2024-12-26',  # Boxing Day
        # 2025
        '2025-01-01',
        '2025-04-18',  # Good Friday
        '2025-04-20',  # Easter Sunday
        '2025-05-01',
        '2025-06-01',
        '2025-10-10',
        '2025-10-20',
        '2025-12-12',
        '2025-12-25',
        '2025-12-26',
        # 2026
        '2026-01-01',
        '2026-04-03',  # Good Friday
        '2026-04-05',  # Easter Sunday
        '2026-05-01',
        '2026-06-01',
        '2026-10-10',
        '2026-10-20',
        '2026-12-12',
        '2026-12-25',
        '2026-12-26',
    ]

    holiday_set = pd.to_datetime(kenya_holidays)
    df['is_holiday'] = df['date'].isin(holiday_set).astype(int)

In [ ]:

if 'is_holiday' in df.columns:
    hol = df.groupby(['is_holiday','business_type'])['units_sold'].mean().reset_index()
    hol['is_holiday'] = hol['is_holiday'].map({0:'Normal day',1:'Public holiday'})

    fig = px.bar(hol, x='business_type', y='units_sold',
        color='is_holiday', barmode='group',
        color_discrete_sequence=['#7F77DD','#D85A30'],
        text=hol['units_sold'].round(1),
        title='Holiday vs normal day: avg units sold by business type',
        labels={'units_sold':'Avg units sold','business_type':'Business type'})
    fig.update_traces(textposition='outside')
    fig.update_layout(height=420)
    fig.show()

---

# Section 4 WHERE: Where in the supply chain does leakage happen?

***Business question***: Which counties, suppliers, and categories are the leakage hotspots?

### 4.1 Shrinkage by supplier county 


In [ ]:

if 'supplier_county' in df.columns:
    county = (df.groupby('supplier_county')
        .agg(total_shrinkage=('shrinkage_value_kes','sum'),
             avg_shrinkage_rate=('shrinkage_rate_7d','mean') if 'shrinkage_rate_7d' in df.columns else ('shrinkage_units','mean'))
        .sort_values('total_shrinkage', ascending=False)
        .reset_index())

    fig = px.bar(county, x='supplier_county', y='total_shrinkage',
        text=county['total_shrinkage'].apply(lambda v: f'KES {v:,.0f}'),
        color='total_shrinkage', color_continuous_scale='Reds',
        title='Total shrinkage loss by supplier county',
        labels={'total_shrinkage':'Total shrinkage (KES)','supplier_county':'Supplier county'})
    fig.update_traces(textposition='outside')
    fig.update_layout(height=500, coloraxis_showscale=False)
    fig.show()

### 4.2 Stock turnover rate by category: where is stock sitting idle? 

In [ ]:

if 'stock_turnover_rate' in df.columns:
    turnover = (df.groupby('category')['stock_turnover_rate']
        .mean().sort_values().reset_index())

    fig = px.bar(turnover, x='stock_turnover_rate', y='category',
        orientation='h',
        color='stock_turnover_rate',
        color_continuous_scale='RdYlGn' ,
        text=turnover['stock_turnover_rate'].round(3),
        title='Avg stock turnover rate by category (higher = faster moving)',
        labels={'stock_turnover_rate':'Avg turnover rate','category':''})
    fig.update_traces(textposition='outside')
    fig.update_layout(height=440, coloraxis_showscale=False)
    fig.add_vline(x=turnover['stock_turnover_rate'].median(),
        line_dash='dash', line_color='gray',
        annotation_text='Median', annotation_position='top')
    fig.show()

### 4.3 Supply chain gap: restock lag analysis 

In [ ]:

# Restock lag = how often closing_stock < reorder_level before restock arrives
if 'needs_restock' in df.columns and 'supplier_lead_days' in df.columns:
    gap = (df[df['needs_restock']==1]
        .groupby(['category','supplier_lead_days'])
        .size().reset_index(name='restock_triggers')
        .sort_values('restock_triggers', ascending=False))

    fig = px.scatter(gap, x='supplier_lead_days', y='restock_triggers',
        size='restock_triggers', color='category',
        title='Restock triggers vs supplier lead time by category',
        labels={'supplier_lead_days':'Lead time (days)','restock_triggers':'Restock trigger events'},
        hover_data=['category'])
    fig.update_layout(height=440)
    fig.show()

    print("\nAverage lead days by category:")
    print(df.groupby('category')['supplier_lead_days'].first().sort_values().to_string())

---

# Section 5 WHY: Why are products running out or losing value?

***Business question***: What are the root causes? Which features drive stockout risk?

### 5.1 Correlation heatmap: what drives stockout risk? 

In [ ]:

df = df.sort_values(['product_id', 'date'])
df['demand_std_7d'] = (
    df.groupby('product_id')['units_sold']
    .transform(lambda x: x.rolling(window=7, min_periods=2).std())
    .round(2)
)

# Step 2: fill any remaining NaNs with product mean
df["demand_std_7d"] = df.groupby("product_id")["demand_std_7d"].transform(
    lambda x: x.fillna(x.mean())
)

# Step 3: if any product has ALL NaNs (only 1 row of data), fill with 0
df["demand_std_7d"] = df["demand_std_7d"].fillna(0)

corr_cols = ['units_sold','shrinkage_units','closing_stock','opening_stock',
             'ma_sales_7d','ma_sales_30d','revenue','demand_std_7d',
             'days_of_stock_remaining','stock_turnover_rate','shrinkage_rate_7d',
             'margin_pct','is_holiday','is_weekend','is_month_end',
             'is_planting_season','stockout_risk']

existing_corr = [c for c in corr_cols if c in df.columns]
corr_matrix = df[existing_corr].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, ax=ax,
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    annot=True, fmt='.2f', annot_kws={'size':8},
    linewidths=0.5, square=True,
    cbar_kws={'label':'Pearson correlation'})
ax.set_title('WHY -- Correlation heatmap: all features vs stockout_risk',
    fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Print top correlates with stockout_risk
if 'stockout_risk' in corr_matrix:
    top = (corr_matrix['stockout_risk']
        .drop('stockout_risk')
        .abs()
        .sort_values(ascending=False)
        .head(8))

    print("\nTop features correlated with stockout_risk:")
    for feat, val in top.items():
        direction = '+' if corr_matrix.loc[feat,'stockout_risk']>0 else '-'
        print(f"  {direction}{val:.3f}  {feat}")

### 5.2 Why does shrinkage happen? margin vs shrinkage rate 


In [ ]:

df.info()

In [ ]:

# Hypothesis: high-margin products are stolen more (pharmacy, electronics)
df = df.sort_values(['product_id', 'date'])

# 7-day rolling shrinkage rate -- smooths daily noise
df['shrinkage_rate_7d'] = (
    df.groupby('product_id')
    .apply(lambda g: g['shrinkage_units'].rolling(7, min_periods=3).sum() /
                      g['opening_stock'].rolling(7, min_periods=3).sum())
    .reset_index(level=0, drop=True)
    .fillna(0).round(4)
)

if 'shrinkage_rate_7d' in df.columns and 'margin_pct' in df.columns:
    shrink_margin = (df.groupby(['product_name','business_type'])
        .agg(avg_margin=('margin_pct','mean'),
             avg_shrinkage=('shrinkage_rate_7d','mean'),
             total_shrinkage_kes=('shrinkage_value_kes','sum'))
        .reset_index())

    fig = px.scatter(shrink_margin,
        x='avg_margin', y='avg_shrinkage',
        size='total_shrinkage_kes',
        color='business_type', color_discrete_map=COLORS,
        hover_name='product_name',
        title='WHY -- Does higher margin = higher shrinkage rate? (bubble = total loss)',
        labels={'avg_margin':'Avg gross margin %',
                'avg_shrinkage':'Avg 7-day shrinkage rate (units/day)'})
    fig.update_layout(height=480)
    fig.show()

    corr_val = shrink_margin[['avg_margin','avg_shrinkage']].corr().iloc[0,1]
    print(f"Correlation between margin and shrinkage rate: {corr_val:.3f}")
    if abs(corr_val) > 0.3:
        print("-> Moderate/strong relationship -- high-margin products are theft-prone")
    else:
        print("-> Weak relationship -- shrinkage is not primarily margin-driven")

### 5.3 Why do stockouts happen? demand acceleration analysis 



In [ ]:

def classify_trend(row):
    if pd.isna(row['ma_sales_7d']) or pd.isna(row['ma_sales_30d']):
        return 'Insufficient Data'
    ratio = row['ma_sales_7d'] / row['ma_sales_30d'] if row['ma_sales_30d'] > 0 else 1.0
    if ratio >= 1.15: return 'Strong Uptrend'
    elif ratio >= 1.05: return 'Uptrend'
    elif ratio <= 0.85: return 'Strong Downtrend'
    elif ratio <= 0.95: return 'Downtrend'
    else: return 'Stable'

df['demand_trend'] = df.apply(classify_trend, axis=1)

trend_order = ['Strong Downtrend', 'Downtrend', 'Stable', 'Uptrend', 'Strong Uptrend']

if 'demand_trend' in df.columns and 'stockout_risk' in df.columns:
    summary = (
        df.groupby('demand_trend')['stockout_risk']
        .agg(total='count', at_risk='sum')
        .assign(pct_at_risk=lambda x: (x['at_risk'] / x['total'] * 100).round(1))
        .reindex(trend_order)
    )

    plot_df =(df.assign(stockout_risk=df['stockout_risk'].map({0:'safe',1:'at risk'}))
        .groupby(['demand_trend', 'stockout_risk'])
        .size().reset_index(name='count'))

    fig = px.bar(
        plot_df,
        x='demand_trend', y='count',
        color='stockout_risk',
        color_discrete_map={'safe': '#7F77DD', 'at risk': '#d62728'},
        barmode='overlay', opacity=0.7,
        category_orders={'demand_trend': trend_order},
        title='Demand trend distribution: stockout risk vs safe days',
        labels={'demand_trend': 'Demand Trend', 'count': 'Number of Records',
                'stockout_risk': 'Stockout Risk'}
    )
    fig.update_layout(height=500,
        legend=dict(title='Stockout Risk', orientation='v', y=1.02))
    fig.show()

    print(summary.to_string())

---

# Section 6 HOW: How can we predict and prevent inventory problems?

***Business question***: Which features are most predictive? How do we calculate the right
reorder quantity?

### 6.1 Days of stock remaining: which products need urgent attention? 


In [ ]:

if 'days_of_stock_remaining' in df.columns:
    # Get latest record per product
    latest = (df.sort_values('date')
        .groupby('product_name')
        .last()
        .reset_index())

    latest['urgency'] = pd.cut(
        latest['days_of_stock_remaining'],
        bins=[-1, 2, 7, 14, 9999],
        labels=['Critical (<2d)','High risk (2-7d)','Monitor (7-8d)','Safe (>7d)']
    )

    fig = px.scatter(latest.dropna(subset=['urgency']),
        x='days_of_stock_remaining', y='product_name',
        color='urgency',
        color_discrete_map={'Critical (<2d)':'#A32D2D','High risk (2-6d)':'#BA7517',
            'Monitor (7-8d)':'#185FA5','Safe (>7d)':'#3B6D11'},
        size_max=12,
        title='Days of stock remaining per product (latest date)',
        labels={'days_of_stock_remaining':'Days remaining','product_name':''})
    fig.add_vline(x=4, line_dash='dash', line_color='red',
        annotation_text='4-day alert threshold')
    fig.update_layout(height=560)
    fig.show()

    summary = latest['urgency'].value_counts()
    print("\nUrgency breakdown:")
    for level, count in summary.items():
        print(f"  {level:25s}: {count} products")

### 6.2 Reorder quantity formula: how much should be ordered? 


In [ ]:

# Economic reorder logic:
# reorder_qty = (avg_daily_demand x lead_days) + safety_stock
# safety_stock = Z x demand_std x sqrt(lead_days)
# Z = 1.65 for 95% service level
SAFETY_FACTOR = 1.65  # 95% service level

if 'ma_sales_7d' in df.columns and 'supplier_lead_days' in df.columns:
    reorder = (df.groupby(['product_name','category','business_type'])
        .agg(avg_daily_demand=('ma_sales_7d','mean'),
             demand_std=('demand_std_7d','mean') if 'demand_std_7d' in df.columns else ('units_sold','std'),
             lead_days=('supplier_lead_days','first'),
             reorder_level=('reorder_level','first'),
             avg_unit_cost=('unit_cost_kes','mean'))
        .reset_index())

    reorder['safety_stock'] = (SAFETY_FACTOR
        * reorder['demand_std'].fillna(0)
        * np.sqrt(reorder['lead_days'])).round(0)

    reorder['recommended_order_qty'] = (
        reorder['avg_daily_demand'] * reorder['lead_days']
        + reorder['safety_stock']
    ).round(0).astype(int)

    reorder['reorder_cost_kes'] = (
        reorder['recommended_order_qty'] * reorder['avg_unit_cost']
    ).round(0)

    display_cols = ['product_name','category','avg_daily_demand',
        'lead_days','safety_stock','recommended_order_qty','reorder_cost_kes']

    print("REORDER QUANTITY RECOMMENDATION TABLE")
    print("=" * 70)
    print(reorder[display_cols].sort_values('reorder_cost_kes', ascending=False)
        .to_string(index=False))

### Key Findings 

In [ ]:

df1 = df.copy()
le = LabelEncoder()
df1['stockout_risk'] = le.fit_transform(df1['stockout_risk'].astype(str))

print("=" * 65)
print(" EDA COMPLETE -- KEY FINDINGS SUMMARY")
print("=" * 65)

findings = []

# WHO
top_biz = df1.groupby('business_type')['shrinkage_value_kes'].sum().idxmax()
findings.append(f"WHO   : '{top_biz}' has the highest shrinkage loss")

# WHAT
total_loss = df1['shrinkage_value_kes'].sum()
findings.append(f"WHAT  : Total shrinkage = KES {total_loss:,.0f}")
top_cat = df.groupby('category')['shrinkage_value_kes'].sum().idxmax()
findings.append(f"WHAT  : Worst category = '{top_cat}'")

# WHEN
if 'is_month_end' in df.columns:
    me_avg = df1[df1['is_month_end']==1]['units_sold'].mean()
    norm_avg= df1[df1['is_month_end']==0]['units_sold'].mean()
    lift = (me_avg/norm_avg - 1)*100
    findings.append(f"WHEN  : Month-end demand is {lift:.0f}% higher than normal days")

# WHERE
if 'stock_turnover_rate' in df.columns:
    slow = df1.groupby('category')['stock_turnover_rate'].mean().idxmin()
    findings.append(f"WHERE : Slowest turnover category = '{slow}' (dead stock risk)")

# WHY
if 'stockout_risk' in df1.columns and 'days_of_stock_remaining' in df1.columns:
    corr_df = df1[['days_of_stock_remaining', 'ma_sales_7d', 'demand_trend', 'stockout_risk']].copy()
    corr_df['demand_trend'] = le.fit_transform(corr_df['demand_trend'].astype(str))
    top_corr = (corr_df.corr()['stockout_risk']
        .drop('stockout_risk').abs().idxmax())
    findings.append(f"WHY   : Strongest predictor of stockout = '{top_corr}'")

# HOW
if 'stockout_risk' in df1.columns:
    risk_pct = df1['stockout_risk'].mean()*100
    findings.append(f"HOW   : {risk_pct:.1f}% of product-days are at stockout risk -> train XGBoost on this")

for f in findings:
    print(f"  {f}")
print("=" * 65)

---

# 7. Feature importance preview: which columns matter most for ML? 

In [ ]:

col = df.select_dtypes(include='object')
for i in col:
    print(f"--- Value Counts for: {i} ---")
    print(df[i].value_counts())
    print("\n")

In [ ]:

df["date"] = pd.to_datetime(df["date"])

# Days since last restock -- most valuable for stockout
df["days_since_restock"] = df.groupby("product_id")["date"].diff().dt.days.fillna(0)

# How far into the month -- demand spikes at month end
df["day_of_month"] = df["date"].dt.day

# Which month -- seasonal demand patterns
df["month"] = df["date"].dt.month

# Which quarter
df["quarter"] = df["date"].dt.quarter

In [ ]:

drop_cols = [
    # target
    "stockout_risk", "risk_score", "risk_label",
    # leakage
    "needs_restock", "days_of_stock_remaining", "recommended_reorder_qty",
    "gross_margin", "margin_pct", "revenue",
    "stock_turnover_rate", "avg_inventory","date","product_id"
]

X = df.drop(columns=drop_cols)
y = df['stockout_risk']

In [ ]:

day_map = {
    'Monday': 0, 'Tuesday': 1, 'Wednesday': 2,
    'Thursday': 3, 'Friday': 4, 'Saturday': 5, 'Sunday': 6
}
X['day_name'] = X['day_name'].map(day_map)

ohe_cols = ['business_type', 'supplier_county', 'demand_trend','unit_of_measure']
le_cols = ['product_name', 'supplier_name', 'category']

#  Step 1: Label encode high-cardinality columns 
for col in le_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

#  Step 2: One-hot encode low-cardinality columns 
X = pd.get_dummies(X, columns=ohe_cols, drop_first=True, dtype=int)

#  Diagnostics 
print(f"Shape after encoding -- train : {X.shape}")
print(f"\nOHE columns created:")
print([c for c in X.columns if any(c.startswith(o) for o in ohe_cols)])

In [ ]:

X.isna().sum()

In [ ]:

sc = StandardScaler()
x_scaled = sc.fit_transform(X)

pca = PCA()
pca.fit(x_scaled)

# Step 3: Auto-select number of components explaining 95% variance
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
n_components = np.argmax(cumulative_variance >= 0.95) + 1
print(f"Components needed for 95% variance: {n_components}")

# Step 4: PFA -- pick the most representative real feature per component
loadings = pca.components_[:n_components]  # shape: (n_components, n_features)
selected_indices = []
for i in range(n_components):
    idx = np.argmax(np.abs(loadings[i]))
    if idx not in selected_indices:
        selected_indices.append(idx)

selected_features = list(X.columns[selected_indices])
print(f"Selected {len(selected_features)} features:", selected_features)

# Step 5: Final dataset for your model
X_selected = X[selected_features]

In [ ]:

SEED = 42
np.random.seed(SEED)

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,stratify=y,random_state=SEED)

In [ ]:

num = X_train.select_dtypes(include=['int','float']).columns
cat = X_train.select_dtypes(include='object').columns
print(num,'\n',cat)

In [ ]:

numeric_cols =['opening_stock', 'units_sold', 'shrinkage_units', 'units_restocked',
    'closing_stock', 'unit_cost_kes', 'selling_price_kes',
    'supplier_lead_days', 'reorder_level',
    'gross_margin', 'margin_pct', 'days_of_stock_remaining', 'ma_sales_7d',
    'stockout_risk', 'risk_score',
    'recommended_reorder_qty', 'ma_sales_30d', 'shrinkage_value_kes',
    'avg_inventory', 'stock_turnover_rate', 'revenue',
    'demand_std_7d', 'shrinkage_rate_7d']

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), num)
], remainder="drop")

# 
# 4. MODEL DEFINITIONS
# 
models = {
    "Logistic Regression": ImbPipeline([
        ("pre", preprocessor),
        ("smote", SMOTE(random_state=SEED)),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED))
    ]),
    "Random Forest": ImbPipeline([
        ("pre", preprocessor),
        ("smote", SMOTE(random_state=SEED)),
        ("clf", RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                        max_depth=10, random_state=SEED, n_jobs=-1))
    ]),
    "XGBoost": ImbPipeline([
        ("pre", preprocessor),
        ("smote", SMOTE(random_state=SEED)),
        ("clf", XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05,
                               scale_pos_weight=3, eval_metric="logloss",
                               random_state=SEED, verbosity=0))
    ]),
    "LightGBM": ImbPipeline([
        ("pre", preprocessor),
        ("smote", SMOTE(random_state=SEED)),
        ("clf", LGBMClassifier(n_estimators=200, max_depth=6, learning_rate=0.05,
                                class_weight="balanced", random_state=SEED, verbose=-1))
    ]),
}

In [ ]:

# 
# 5. TRAINING, CROSS-VALIDATION & EVALUATION
# 
print("\n[3/6] Training & evaluating models...")
print("-" * 60)

CV_FOLDS = 5
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
results = {}

for name, pipeline in models.items():
    print(f"\n  {name}")

    # CV on training set
    cv_scores = cross_val_score(pipeline, X_train, y_train,
                                 cv=cv, scoring="roc_auc", n_jobs=-1)
    print(f"    CV ROC-AUC: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")

    # Fit on full train set
    pipeline.fit(X_train, y_train)

    # Predict
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    roc_auc = roc_auc_score(y_test, y_prob)
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    avg_prec = average_precision_score(y_test, y_prob)

    results[name] = {
        "pipeline": pipeline,
        "cv_roc_auc": cv_scores.mean(),
        "cv_std": cv_scores.std(),
        "test_roc_auc": roc_auc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "avg_precision": avg_prec,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }

    print(f"    Test ROC-AUC: {roc_auc:.4f} | F1: {f1:.4f} | "
          f"Precision: {precision:.4f} | Recall: {recall:.4f}")
    print(f"    Avg Precision: {avg_prec:.4f}")

# 
# 6. SELECT BEST MODEL
# 
print("\n[4/6] Selecting best model...")
best_name = max(results, key=lambda k: results[k]["test_roc_auc"])
best = results[best_name]
print(f"  Best model: {best_name} (ROC-AUC = {best['test_roc_auc']:.4f})")

print(f"\n  Classification Report -- {best_name}:")
print(classification_report(y_test, best["y_pred"],
                             target_names=["No Risk", "Stock-Out Risk"]))

# 
# 7. VISUALIZATIONS
# 
print("[5/6] Generating evaluation plots...")

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle("Stock-Out Risk Classification -- Model Evaluation", fontsize=15, fontweight="bold")

# --- (a) Model comparison bar chart ---
ax = axes[0, 0]
model_names = list(results.keys())
roc_scores = [results[m]["test_roc_auc"] for m in model_names]
f1_scores = [results[m]["f1"] for m in model_names]
x = np.arange(len(model_names))
width = 0.35
bars1 = ax.bar(x - width/2, roc_scores, width, label="ROC-AUC", color="#2196F3", alpha=0.85)
bars2 = ax.bar(x + width/2, f1_scores, width, label="F1 Score", color="#4CAF50", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([m.replace(" ", "\n") for m in model_names], fontsize=9)
ax.set_ylim(0, 1.1)
ax.set_title("Model Comparison")
ax.set_ylabel("Score")
ax.legend()
ax.bar_label(bars1, fmt="%.3f", fontsize=8)
ax.bar_label(bars2, fmt="%.3f", fontsize=8)

# --- (b) ROC curves for all models ---
ax = axes[0, 1]
for name, res in results.items():
    RocCurveDisplay.from_predictions(y_test, res["y_prob"], name=name, ax=ax)
ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
ax.set_title("ROC Curves (All Models)")
ax.legend(fontsize=8)

# --- (c) Precision-Recall curve (best model) ---
ax = axes[0, 2]
prec, rec, _ = precision_recall_curve(y_test, best["y_prob"])
ap = best["avg_precision"]
ax.plot(rec, prec, color="#E91E63", lw=2, label=f"AP = {ap:.3f}")
ax.fill_between(rec, prec, alpha=0.1, color="#E91E63")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title(f"Precision-Recall Curve\n({best_name})")
ax.legend()

# --- (d) Confusion matrix (best model) ---
ax = axes[1, 0]
cm = confusion_matrix(y_test, best["y_pred"])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
    xticklabels=["No Risk", "Stock-Out"], yticklabels=["No Risk", "Stock-Out"])
ax.set_title(f"Confusion Matrix\n({best_name})")
ax.set_ylabel("Actual")
ax.set_xlabel("Predicted")

# --- (e) Feature importance (best model -- Random Forest / tree-based) ---
ax = axes[1, 1]
try:
    clf = best["pipeline"].named_steps["clf"]
    pre = best["pipeline"].named_steps["pre"]
    feat_names = cat + num
    if hasattr(clf, "feature_importances_"):
        importances = clf.feature_importances_
        indices = np.argsort(importances)[-15:]
        ax.barh([feat_names[i] for i in indices], importances[indices],
                color="#FF9800", alpha=0.85)
        ax.set_title(f"Top 15 Feature Importances\n({best_name})")
        ax.set_xlabel("Importance")
    else:
        ax.text(0.5, 0.5, "Not available\nfor this model type",
                ha="center", va="center", transform=ax.transAxes)
        ax.set_title("Feature Importances")
except Exception:
    ax.set_title("Feature Importances (N/A)")

# --- (f) CV score distribution ---
ax = axes[1, 2]
cv_means = [results[m]["cv_roc_auc"] for m in model_names]
cv_stds = [results[m]["cv_std"] for m in model_names]
colors = ["#2196F3", "#4CAF50", "#FF5722", "#9C27B0"]
ax.barh(model_names, cv_means, xerr=cv_stds, color=colors, alpha=0.8, capsize=5)
ax.set_xlabel("CV ROC-AUC")
ax.set_title(f"Cross-Validation Scores ({CV_FOLDS}-Fold)")
ax.set_xlim(0.5, 1.0)
for i, (m, s) in enumerate(zip(cv_means, cv_stds)):
    ax.text(m + 0.005, i, f"{m:.3f}+/-{s:.3f}", va="center", fontsize=8)

plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, "model_evaluation.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"  Plot saved -> {plot_path}")

# 
# 8. SAVE BEST MODEL + METADATA
# 
print("[6/6] Saving model artifacts...")

# Save pipeline (includes preprocessor + SMOTE + classifier)
# model_path = os.path.join(OUTPUT_DIR, "stockout_model.joblib")
# joblib.dump(best["pipeline"], model_path)

pipeline = best["pipeline"]
clf = pipeline.named_steps["clf"]

if isinstance(clf, (XGBClassifier)):
    # Save the XGBoost model in native format
    model_path = os.path.join(OUTPUT_DIR, "stockout_model.json")
    clf.save_model(model_path)

    # Save the preprocessing + SMOTE pipeline separately
    preprocessor_path = os.path.join(OUTPUT_DIR, "stockout_preprocessor.joblib")
    joblib.dump(pipeline[:-1], preprocessor_path)
else:
    # Save the complete pipeline for non-XGBoost models
    model_path = os.path.join(OUTPUT_DIR, "stockout_model.joblib")
    joblib.dump(pipeline, model_path)

# Save metadata for deployment
metadata = {
    "model_name": best_name,
    "saved_at": datetime.now().isoformat(),
    "target": y,
    "numeric_features": num,
    "categorical_features": cat,
    "test_metrics": {
        "roc_auc": round(best["test_roc_auc"], 4),
        "f1": round(best["f1"], 4),
        "precision": round(best["precision"], 4),
        "recall": round(best["recall"], 4),
        "avg_precision": round(best["avg_precision"], 4),
    },
    "cv_metrics": {
        "mean_roc_auc": round(best["cv_roc_auc"], 4),
        "std_roc_auc": round(best["cv_std"], 4),
    },
    "all_model_results": {
        name: {
            "test_roc_auc": round(res["test_roc_auc"], 4),
            "f1": round(res["f1"], 4),
        }
        for name, res in results.items()
    },
    "class_labels": {"0": "No Stock-Out Risk", "1": "Stock-Out Risk"},
    "currency": "KES",
}

meta_path = os.path.join(OUTPUT_DIR, "model_metadata.json")
with open(meta_path, "w") as f:
    json.dump(list(metadata), f, indent=2)

print(f"  Model saved -> {model_path}")
print(f"  Metadata   -> {meta_path}")

In [ ]:

import joblib

bundle = {
    "model":best["pipeline"],
    "features":selected_features,
    "scaler":StandardScaler(),
    "X_test": X_test,
    "y_test": y_test
}

joblib.dump(bundle,"model_bundle.joblib")

In [ ]:
# Cell [123]
bundle = joblib.load("model_bundle.joblib")
model = bundle['model']
X_test = bundle['X_test']
y_test = bundle['y_test']

prediction = model.predict(X_test)
print("Prediction: ", prediction[:10])

In [ ]:

print(classification_report(y_test,prediction))
print(confusion_matrix(y_test,prediction))